# Forecasting and Model Validation – Daily Net Revenue

This notebook runs the complete forecasting pipeline in `forecasting_pipeline.py`.

Models:
- Naive baseline
- Seven-day seasonal-naive baseline
- Exponential Smoothing (weekly additive seasonality is used only when the training data has at least three weeks and lag-7 ACF is at least 0.20)
- `HistGradientBoostingRegressor` with calendar, lag and rolling features

Evaluation:
- Three expanding-window chronological validation folds, each 14 days
- A separate untouched final test period of 28 days
- MAE and RMSE
- All models forecast exactly the same dates within each fold/test period
- The final seven-day forecast is produced after refitting on all observed data

The HistGradientBoosting forecast is recursive: each forecast is appended to the history before constructing the next day's lag and rolling features.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Keep forecasting_pipeline.py in the same folder as this notebook.
pipeline_path = Path("forecasting_pipeline.py")
if not pipeline_path.exists():
    raise FileNotFoundError("Place forecasting_pipeline.py in the notebook folder.")

namespace = {}
exec(compile(pipeline_path.read_text(), str(pipeline_path), "exec"), namespace)
run_pipeline = namespace["run_pipeline"]

## 1. Run the pipeline

In [ ]:
DATA_DIR = Path(".")  # Change if the CSV files are elsewhere.
OUTPUT_DIR = Path("forecast_outputs")

comparison, forecast_table, final_test_metrics = run_pipeline(
    DATA_DIR / "daily_net_revenue.csv",
    DATA_DIR / "sales_transactions.csv",
    OUTPUT_DIR
)

## 2. Inspect model comparison and final test performance

In [ ]:
display(comparison.sort_values("Validation_MAE"))
display(final_test_metrics.sort_values("MAE"))

## 3. Inspect the seven-day forecast

In [ ]:
display(forecast_table.sort_values(["Model", "Date"]))
forecast_table.to_csv(OUTPUT_DIR / "seven_day_forecast.csv", index=False)

## 4. Output files

The `forecast_outputs` folder contains:
- `prepared_daily_series.csv`
- `transaction_reconciliation.csv`
- `validation_model_comparison.csv`
- `validation_forecasts.csv`
- `final_test_model_comparison.csv`
- `final_test_forecasts.csv`
- `model_comparison_summary.csv`
- `seven_day_forecast.csv` (long format)
- `seven_day_forecast_wide.csv` (one row per date, one column per model)

No random train/test split is used. The final test set is chronological and is not used to fit any model.